In [1]:
# =========================
# 1. IMPORTS
# =========================
import pandas as pd
import numpy as np
import json
from pathlib import Path

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)

In [2]:
# =========================
# 2. FILE PATHS
# =========================
BASE_DIR = Path("..")
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

EMS_FILE = DATA_DIR / "ems_calls_2024_2025.csv"
WEATHER_FILE = DATA_DIR / "noaa_weather_sf.csv"
HHS_FILE = DATA_DIR / "hhs_hospital_capacity.csv"
FEMA_FILE = DATA_DIR / "fema_disasters_ca.csv"

print(EMS_FILE)
print(WEATHER_FILE)
print(HHS_FILE)
print(FEMA_FILE)

..\data\ems_calls_2024_2025.csv
..\data\noaa_weather_sf.csv
..\data\hhs_hospital_capacity.csv
..\data\fema_disasters_ca.csv


In [3]:
# =========================
# 3. LOAD DATA
# =========================
ems = pd.read_csv(EMS_FILE)
weather = pd.read_csv(WEATHER_FILE)
hhs = pd.read_csv(HHS_FILE, low_memory=False)
fema = pd.read_csv(FEMA_FILE)

print("EMS shape:", ems.shape)
print("Weather shape:", weather.shape)
print("HHS shape:", hhs.shape)
print("FEMA shape:", fema.shape)

EMS shape: (488349, 36)
Weather shape: (695, 13)
HHS shape: (74549, 128)
FEMA shape: (1686, 28)


In [4]:
# =========================
# 4. QUICK INSPECTION
# =========================
def inspect_df(name, df, n=5):
    print(f"\n===== {name} =====")
    print(df.head(n))
    print("\nColumns:")
    print(df.columns.tolist())

inspect_df("EMS", ems)
inspect_df("WEATHER", weather)
inspect_df("HHS", hhs)
inspect_df("FEMA", fema)


===== EMS =====
   Call Number Unit ID  Incident Number         Call Type   Call Date  Watch Date            Received DtTm               Entry DtTm            Dispatch DtTm            Response DtTm  \
0    240010774    M571         24000149  Medical Incident  01/01/2024  12/31/2023  2024 Jan 01 05:55:16 AM  2024 Jan 01 05:56:25 AM  2024 Jan 01 06:05:57 AM  2024 Jan 01 06:06:03 AM   
1    240010644     E33         24000122  Medical Incident  01/01/2024  12/31/2023  2024 Jan 01 04:10:02 AM  2024 Jan 01 04:10:46 AM  2024 Jan 01 04:13:33 AM  2024 Jan 01 04:15:14 AM   
2    240010774     E05         24000149  Medical Incident  01/01/2024  12/31/2023  2024 Jan 01 05:55:16 AM  2024 Jan 01 05:56:25 AM  2024 Jan 01 05:56:56 AM  2024 Jan 01 05:58:21 AM   
3    240011849     T15         24000348  Medical Incident  01/01/2024  01/01/2024  2024 Jan 01 02:19:16 PM  2024 Jan 01 02:20:07 PM  2024 Jan 01 02:20:51 PM  2024 Jan 01 02:22:47 PM   
4    240010925    M591         24000182  Medical Incident 

In [5]:
# =========================
# 5. HELPERS
# =========================
def find_first_matching_column(df, candidates):
    cols_lower = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in cols_lower:
            return cols_lower[cand.lower()]
    return None

def parse_date_column(df, candidates, new_name="date"):
    col = find_first_matching_column(df, candidates)
    if col is None:
        raise ValueError(f"Could not find any of these date columns: {candidates}")
    out = df.copy()
    out[new_name] = pd.to_datetime(out[col], errors="coerce")
    return out, col

def safe_numeric(series):
    return pd.to_numeric(series, errors="coerce")

In [6]:
# =========================
# 6. EMS CLEANING
# =========================
# Try common possible date column names for EMS
ems_date_candidates = [
    "call_date", "Call Date", "calldate", "date", "incident_date", "received_dttm"
]

ems, ems_date_col = parse_date_column(ems, ems_date_candidates, new_name="date")
print("EMS date column used:", ems_date_col)

# Keep only rows with valid dates
ems = ems.dropna(subset=["date"]).copy()
ems["date_only"] = ems["date"].dt.date

# Daily EMS volume
ems_daily = (
    ems.groupby("date_only")
       .size()
       .reset_index(name="ems_calls")
       .rename(columns={"date_only": "date"})
)

ems_daily["date"] = pd.to_datetime(ems_daily["date"])
ems_daily.head()

EMS date column used: Call Date


,date,ems_calls
0,2024-01-01,801
1,2024-01-02,671
2,2024-01-03,648
3,2024-01-04,776
4,2024-01-05,726


In [7]:
weather.columns

Index(['STATION', 'NAME', 'LATITUDE', 'LONGITUDE', 'ELEVATION', 'DATE', 'PRCP', 'SNWD', 'TAVG', 'TMAX', 'TMIN', 'TOBS', 'WT01'], dtype='object')

In [8]:
# =========================
# 7. WEATHER CLEANING
# =========================
weather_date_candidates = ["DATE", "date", "Date"]
weather, weather_date_col = parse_date_column(weather, weather_date_candidates, new_name="date")
print("Weather date column used:", weather_date_col)

weather = weather.dropna(subset=["date"]).copy()

# Try common weather columns
tmax_col = find_first_matching_column(weather, ["TMAX", "tmax", "temp_max", "maximum temperature"])
tmin_col = find_first_matching_column(weather, ["TMIN", "tmin", "temp_min", "minimum temperature"])
prcp_col = find_first_matching_column(weather, ["PRCP", "prcp", "precipitation", "rain"])

print("Weather columns used:")
print("TMAX:", tmax_col)
print("TMIN:", tmin_col)
print("PRCP:", prcp_col)

weather_daily = pd.DataFrame({"date": pd.to_datetime(weather["date"].dt.date)})

if tmax_col:
    weather_daily["tmax"] = safe_numeric(weather[tmax_col])
else:
    weather_daily["tmax"] = np.nan

if tmin_col:
    weather_daily["tmin"] = safe_numeric(weather[tmin_col])
else:
    weather_daily["tmin"] = np.nan

if prcp_col:
    weather_daily["prcp"] = safe_numeric(weather[prcp_col])
else:
    weather_daily["prcp"] = np.nan

# If NOAA values are in tenths, uncomment this:
# weather_daily["tmax"] = weather_daily["tmax"] / 10
# weather_daily["tmin"] = weather_daily["tmin"] / 10
# weather_daily["prcp"] = weather_daily["prcp"] / 10

weather_daily.head()

Weather date column used: DATE
Weather columns used:
TMAX: TMAX
TMIN: TMIN
PRCP: PRCP


,date,tmax,tmin,prcp
0,2024-01-01,59.0,48.0,0.00
1,2024-01-02,61.0,48.0,0.31
2,2024-01-03,56.0,48.0,0.05
3,2024-01-04,61.0,47.0,0.00
4,2024-01-05,62.0,50.0,0.00


In [9]:
# =========================
# 8. HHS CLEANING
# =========================
hhs_date_candidates = [
    "date", "Date", "collection_week", "weekendingdate", "hospital_pk", "record_date"
]

# If no valid date column exists, we will not fail the notebook
try:
    hhs, hhs_date_col = parse_date_column(hhs, hhs_date_candidates, new_name="date")
    print("HHS date column used:", hhs_date_col)
except Exception as e:
    print("HHS date parsing issue:", e)
    hhs["date"] = pd.NaT

# Try common capacity columns
staffed_beds_col = find_first_matching_column(hhs, [
    "inpatient_beds_used", "total_staffed_adult_icu_beds", "staffed_beds",
    "total_staffed_beds", "all_adult_hospital_beds", "total_beds"
])

occupied_beds_col = find_first_matching_column(hhs, [
    "inpatient_beds_used", "beds_occupied", "occupied_beds", "hospitalized_patients",
    "all_adult_hospital_inpatient_beds_occupied"
])

print("HHS columns used:")
print("Staffed beds column:", staffed_beds_col)
print("Occupied beds column:", occupied_beds_col)

if hhs["date"].notna().sum() > 0:
    hhs = hhs.dropna(subset=["date"]).copy()
    hhs["date_only"] = hhs["date"].dt.date

    if staffed_beds_col and occupied_beds_col:
        hhs["staffed_beds_num"] = safe_numeric(hhs[staffed_beds_col])
        hhs["occupied_beds_num"] = safe_numeric(hhs[occupied_beds_col])

        # If the same column got matched for both, utilization becomes 1.0, so handle that
        if staffed_beds_col == occupied_beds_col:
            hhs["hospital_utilization"] = np.nan
        else:
            hhs["hospital_utilization"] = hhs["occupied_beds_num"] / hhs["staffed_beds_num"]
    else:
        hhs["hospital_utilization"] = np.nan

    hhs_daily = (
        hhs.groupby("date_only", as_index=False)["hospital_utilization"]
           .mean()
           .rename(columns={"date_only": "date"})
    )
    hhs_daily["date"] = pd.to_datetime(hhs_daily["date"])
else:
    hhs_daily = pd.DataFrame(columns=["date", "hospital_utilization"])

hhs_daily.head()

HHS date column used: collection_week
HHS columns used:
Staffed beds column: None
Occupied beds column: None


,date,hospital_utilization
0,2020-02-02,NaN
1,2020-03-01,NaN
2,2020-03-08,NaN
3,2020-03-15,NaN
4,2020-03-22,NaN


In [10]:
# =========================
# 9. FEMA CLEANING
# =========================
# FEMA usually has event begin / end dates
begin_col = find_first_matching_column(fema, [
    "incidentBeginDate", "incidentbegindate", "beginDate", "start_date", "declarationDate"
])
end_col = find_first_matching_column(fema, [
    "incidentEndDate", "incidentenddate", "endDate", "end_date"
])

print("FEMA begin column:", begin_col)
print("FEMA end column:", end_col)

fema_clean = fema.copy()

if begin_col:
    fema_clean["begin_date"] = pd.to_datetime(fema_clean[begin_col], errors="coerce")
else:
    fema_clean["begin_date"] = pd.NaT

if end_col:
    fema_clean["end_date"] = pd.to_datetime(fema_clean[end_col], errors="coerce")
else:
    fema_clean["end_date"] = fema_clean["begin_date"]

# Keep valid rows
fema_clean = fema_clean.dropna(subset=["begin_date"]).copy()
fema_clean["end_date"] = fema_clean["end_date"].fillna(fema_clean["begin_date"])

# Expand event ranges into daily flags
fema_rows = []
for _, row in fema_clean.iterrows():
    try:
        rng = pd.date_range(row["begin_date"].date(), row["end_date"].date(), freq="D")
        for d in rng:
            fema_rows.append({"date": pd.to_datetime(d), "disaster_flag": 1})
    except Exception:
        continue

fema_daily = pd.DataFrame(fema_rows)

if not fema_daily.empty:
    fema_daily = fema_daily.groupby("date", as_index=False)["disaster_flag"].max()
else:
    fema_daily = pd.DataFrame(columns=["date", "disaster_flag"])

fema_daily.head()

FEMA begin column: incidentBeginDate
FEMA end column: incidentEndDate


,date,disaster_flag
0,1954-02-05,1
1,1955-12-23,1
2,1956-12-29,1
3,1958-04-04,1
4,1961-11-16,1


In [11]:
# =========================
# 10. MERGE DAILY TABLE
# =========================
daily = ems_daily.copy()

daily = daily.merge(weather_daily, on="date", how="left")
daily = daily.merge(hhs_daily, on="date", how="left")
daily = daily.merge(fema_daily, on="date", how="left")

daily["disaster_flag"] = daily["disaster_flag"].fillna(0).astype(int)

# Fill hospital utilization with rolling or fallback if missing
if "hospital_utilization" in daily.columns:
    daily["hospital_utilization"] = daily["hospital_utilization"].fillna(method="ffill")
    daily["hospital_utilization"] = daily["hospital_utilization"].fillna(daily["hospital_utilization"].mean())

# Simple heat risk rule
if "tmax" in daily.columns:
    daily["heat_risk"] = daily["tmax"] >= 30
else:
    daily["heat_risk"] = False

daily.head(10)

C:\Users\58in\AppData\Local\Temp\ipykernel_33340\883522800.py:14: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  daily["hospital_utilization"] = daily["hospital_utilization"].fillna(method="ffill")


,date,ems_calls,tmax,tmin,prcp,hospital_utilization,disaster_flag,heat_risk
0,2024-01-01,801,59.0,48.0,0.00,NaN,0,True
1,2024-01-02,671,61.0,48.0,0.31,NaN,0,True
2,2024-01-03,648,56.0,48.0,0.05,NaN,0,True
3,2024-01-04,776,61.0,47.0,0.00,NaN,0,True
4,2024-01-05,726,62.0,50.0,0.00,NaN,0,True
5,2024-01-06,720,55.0,47.0,0.10,NaN,0,True
6,2024-01-07,614,55.0,42.0,0.00,NaN,0,True
7,2024-01-08,814,56.0,44.0,0.00,NaN,0,True
8,2024-01-09,629,57.0,48.0,0.25,NaN,0,True
9,2024-01-10,653,56.0,45.0,0.32,NaN,0,True


In [12]:
# =========================
# 11. SAVE MERGED TABLE
# =========================
merged_path = OUTPUT_DIR / "daily_merged.csv"
daily.to_csv(merged_path, index=False)
print("Saved:", merged_path)

Saved: ..\outputs\daily_merged.csv


In [13]:
# =========================
# 12. BUILD SITUATION OBJECT
# =========================
latest = daily.sort_values("date").iloc[-1]

situation = {
    "date": str(latest["date"].date()),
    "ems_calls_today": int(latest["ems_calls"]) if pd.notna(latest["ems_calls"]) else 0,
    "tmax": float(latest["tmax"]) if "tmax" in daily.columns and pd.notna(latest["tmax"]) else None,
    "prcp": float(latest["prcp"]) if "prcp" in daily.columns and pd.notna(latest["prcp"]) else None,
    "hospital_utilization": float(latest["hospital_utilization"]) if "hospital_utilization" in daily.columns and pd.notna(latest["hospital_utilization"]) else None,
    "disaster_flag": int(latest["disaster_flag"]) if pd.notna(latest["disaster_flag"]) else 0,
    "heat_risk": bool(latest["heat_risk"]),
}

print(json.dumps(situation, indent=2))

{
  "date": "2025-12-31",
  "ems_calls_today": 686,
  "tmax": 53.0,
  "prcp": 0.54,
  "hospital_utilization": null,
  "disaster_flag": 0,
  "heat_risk": true
}


In [14]:
# =========================
# 13. AGENT 1 - FORECAST AGENT
# =========================
def forecast_agent(situation_dict):
    base = situation_dict["ems_calls_today"]

    if situation_dict["heat_risk"]:
        base = int(base * 1.20)

    if situation_dict["disaster_flag"] == 1:
        base = int(base * 1.15)

    util = situation_dict.get("hospital_utilization")
    if util is not None and util > 0.90:
        base = int(base * 1.10)

    return base

predicted_surge = forecast_agent(situation)
print("Predicted surge:", predicted_surge)

Predicted surge: 823


In [15]:
# =========================
# 14. AGENT 2 - PLANNER AGENT
# =========================
def planner_agent(situation_dict, forecast_value):
    actions = []

    util = situation_dict.get("hospital_utilization")

    if forecast_value > 200:
        actions.append("Open emergency surge unit")

    if util is not None and util > 0.85:
        actions.append("Divert non-critical patients to nearby hospitals")

    if situation_dict["heat_risk"]:
        actions.append("Open cooling centers and issue heat advisory")

    if situation_dict["disaster_flag"] == 1:
        actions.append("Activate inter-agency emergency coordination")

    if not actions:
        actions.append("Continue normal monitoring")

    return actions

actions = planner_agent(situation, predicted_surge)
print(actions)

['Open emergency surge unit', 'Open cooling centers and issue heat advisory']


In [16]:
final_state = {
    "date": str(situation["date"]),
    "ems_calls": int(situation["ems_calls_today"]),
    "predicted_surge": int(predicted_surge),
    "heat_risk": bool(situation["heat_risk"]),
    "hospital_utilization": situation.get("hospital_utilization"),
    "disaster_flag": int(situation["disaster_flag"]),
    "actions": actions
}

print(final_state)

{'date': '2025-12-31', 'ems_calls': 686, 'predicted_surge': 823, 'heat_risk': True, 'hospital_utilization': None, 'disaster_flag': 0, 'actions': ['Open emergency surge unit', 'Open cooling centers and issue heat advisory']}


In [17]:
import json
from pathlib import Path

output_path = Path("../outputs/situation.json")

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(final_state, f, indent=2)

print(f"Saved to: {output_path}")

Saved to: ..\outputs\situation.json


In [18]:
prompt = f"""
You are an emergency response assistant.

Current crisis situation:
- Date: {final_state['date']}
- EMS calls: {final_state['ems_calls']}
- Predicted surge: {final_state['predicted_surge']}
- Heat risk: {final_state['heat_risk']}
- Hospital utilization: {final_state['hospital_utilization']}
- Disaster flag: {final_state['disaster_flag']}

Recommended actions:
- {"; ".join(final_state["actions"])}

Write:
1. a short hospital admin alert
2. a short public advisory
"""

print(prompt)


You are an emergency response assistant.

Current crisis situation:
- Date: 2025-12-31
- EMS calls: 686
- Predicted surge: 823
- Heat risk: True
- Hospital utilization: None
- Disaster flag: 0

Recommended actions:
- Open emergency surge unit; Open cooling centers and issue heat advisory

Write:
1. a short hospital admin alert
2. a short public advisory



In [19]:
prompt = f"""
You are an emergency response assistant for a city crisis command center.

Current crisis situation:
- Date: {final_state['date']}
- EMS calls today: {final_state['ems_calls']}
- Predicted surge: {final_state['predicted_surge']}
- Heat risk: {final_state['heat_risk']}
- Hospital utilization: {final_state['hospital_utilization']}
- Disaster flag: {final_state['disaster_flag']}

Recommended actions:
- {"; ".join(final_state["actions"])}

Write the output in exactly this format:

HOSPITAL ADMIN ALERT
- bullet 1
- bullet 2
- bullet 3

PUBLIC ADVISORY
- bullet 1
- bullet 2
- bullet 3

Keep it short, clear, practical, and professional.
"""
print(prompt)


You are an emergency response assistant for a city crisis command center.

Current crisis situation:
- Date: 2025-12-31
- EMS calls today: 686
- Predicted surge: 823
- Heat risk: True
- Hospital utilization: None
- Disaster flag: 0

Recommended actions:
- Open emergency surge unit; Open cooling centers and issue heat advisory

Write the output in exactly this format:

HOSPITAL ADMIN ALERT
- bullet 1
- bullet 2
- bullet 3

PUBLIC ADVISORY
- bullet 1
- bullet 2
- bullet 3

Keep it short, clear, practical, and professional.



In [20]:
PROJECT_ID = "851e1f3c-79bf-442f-9dcb-8dfd2337f0a9"
WATSONX_URL = "https://us-south.ml.cloud.ibm.com"

# paste your current key only temporarily for this local test
API_KEY = "Z-L_PjbvWV4MHS87gZ5ceTaSgHG-IJB7Kyw-QxmZKJNj"

In [21]:
import requests
import json

In [22]:
def get_ibm_token(api_key):
    url = "https://iam.cloud.ibm.com/identity/token"
    headers = {
        "Content-Type": "application/x-www-form-urlencoded"
    }
    data = f"grant_type=urn:ibm:params:oauth:grant-type:apikey&apikey={api_key}"

    response = requests.post(url, headers=headers, data=data, timeout=30)
    response.raise_for_status()
    return response.json()["access_token"]

token = get_ibm_token(API_KEY)
print("IBM token created successfully")

IBM token created successfully


In [23]:
def ask_watsonx(prompt, token, project_id, watsonx_url):
    url = f"{watsonx_url}/ml/v1/text/generation?version=2023-05-29"

    headers = {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json",
        "Accept": "application/json"
    }

    body = {
        "input": prompt,
        "parameters": {
            "decoding_method": "greedy",
            "max_new_tokens": 300,
            "min_new_tokens": 20
        },
        "model_id": "ibm/granite-3-8b-instruct",
        "project_id": project_id
    }

    response = requests.post(url, headers=headers, json=body, timeout=60)
    response.raise_for_status()
    return response.json()

In [24]:
result = ask_watsonx(prompt, token, PROJECT_ID, WATSONX_URL)
print(json.dumps(result, indent=2))

{
  "model_id": "ibm/granite-3-8b-instruct",
  "model_version": "1.1.0",
  "created_at": "2026-04-21T01:52:21.331Z",
  "results": [
    {
      "generated_text": "\nHOSPITAL ADMIN ALERT\n- Open emergency surge unit to prepare for increased patient influx due to predicted surge in EMS calls.\n- Mobilize medical staff to manage heat-related illnesses.\n- Coordinate with local clinics and private practitioners for additional resources.\n\nPUBLIC ADVISORY\n- Heat advisory issued for today: Stay hydrated, limit outdoor activities, and wear light clothing.\n- Cooling centers are open across the city; check local listings for location and hours.\n- Elderly, children, and those with chronic illnesses are at higher risk; please check on vulnerable neighbors.",
      "generated_token_count": 147,
      "input_token_count": 175,
      "stop_reason": "eos_token"
    }
  ],
  "system": {
    "warnings": [
      {
        "message": "Model 'ibm/granite-3-8b-instruct' is in withdrawn state from 2026-

In [25]:
generated_text = result["results"][0]["generated_text"]
print(generated_text)


HOSPITAL ADMIN ALERT
- Open emergency surge unit to prepare for increased patient influx due to predicted surge in EMS calls.
- Mobilize medical staff to manage heat-related illnesses.
- Coordinate with local clinics and private practitioners for additional resources.

PUBLIC ADVISORY
- Heat advisory issued for today: Stay hydrated, limit outdoor activities, and wear light clothing.
- Cooling centers are open across the city; check local listings for location and hours.
- Elderly, children, and those with chronic illnesses are at higher risk; please check on vulnerable neighbors.


In [26]:
ai_output = {
    "prompt": prompt,
    "generated_text": generated_text
}

with open("../outputs/watsonx_output.json", "w", encoding="utf-8") as f:
    json.dump(ai_output, f, indent=2)

print("Saved watsonx output")

Saved watsonx output
